In [0]:
from pyspark.sql.functions import year, month, sum as _sum, rank
from pyspark.sql.window import Window

In [0]:
# 4a. Monthly revenue by region
spark.sql("""
  CREATE OR REPLACE VIEW de_workspace26.ecommerce_badal.gold_monthly_revenue_by_region AS
  SELECT
    YEAR(order_date)  AS year,
    MONTH(order_date) AS month,
    region,
    SUM(revenue)      AS total_revenue
  FROM de_workspace26.ecommerce_badal.silver_orders
  GROUP BY 1, 2, 3
  ORDER BY 1, 2, 3
""")

DataFrame[]

In [0]:
print("gold_monthly_revenue_by_region view created")
spark.sql("SELECT * FROM de_workspace26.ecommerce_badal.gold_monthly_revenue_by_region").display()

gold_monthly_revenue_by_region view created


year,month,region,total_revenue
2024,1,East,13993.0
2024,1,North,454868.0
2024,1,South,300663.0
2024,1,West,5097.0
2024,2,East,190961.0
2024,2,North,155588.0
2024,2,South,64170.0
2024,2,West,787970.0
2024,3,East,280474.0
2024,3,North,248977.0


In [0]:
# 4b. Top products by revenue per category using RANK()
spark.sql("""
  CREATE OR REPLACE VIEW de_workspace26.ecommerce_badal.gold_top_products AS
  SELECT
    product_id,
    product_name,
    category,
    SUM(revenue)                                                    AS total_revenue,
    RANK() OVER (PARTITION BY category ORDER BY SUM(revenue) DESC) AS revenue_rank
  FROM de_workspace26.ecommerce_badal.silver_orders
  GROUP BY product_id, product_name, category
""")

print("gold_top_products view created")
spark.sql("""
  SELECT * FROM de_workspace26.ecommerce_badal.gold_top_products
  ORDER BY category, revenue_rank
""").display()

gold_top_products view created


product_id,product_name,category,total_revenue,revenue_rank
PROD09,Python Programming Book,Books,51397.0,1
PROD02,Denim Jacket,Clothing,64457.0,1
PROD01,Laptop Pro,Electronics,2599948.0,1
PROD07,Smartphone,Electronics,2239936.0,2
PROD04,Wireless Earbuds,Electronics,262425.0,3
PROD03,Coffee Maker,Home,110963.0,1
PROD06,Blender,Home,52171.0,2
PROD10,Table Lamp,Home,31465.0,3
PROD05,Running Shoes,Sports,140736.0,1
PROD08,Yoga Mat,Sports,46833.0,2


In [0]:
# 4c. Customer lifetime value — total revenue per customer
spark.sql("""
  CREATE OR REPLACE VIEW de_workspace26.ecommerce_badal.gold_customer_ltv AS
  SELECT
    o.customer_id,
    c.name,
    c.city,
    c.loyalty_tier,
    COUNT(DISTINCT o.order_id)  AS total_orders,
    SUM(o.revenue)              AS lifetime_value,
    ROUND(AVG(o.revenue), 2)    AS avg_order_value
  FROM de_workspace26.ecommerce_badal.silver_orders o
  LEFT JOIN de_workspace26.ecommerce_badal.silver_customers c
    ON o.customer_id = c.customer_id
   AND c.is_current = true
  GROUP BY o.customer_id, c.name, c.city, c.loyalty_tier
  ORDER BY lifetime_value DESC
""")

print("gold_customer_ltv view created")
spark.sql("SELECT * FROM de_workspace26.ecommerce_badal.gold_customer_ltv").display()

gold_customer_ltv view created


customer_id,name,city,loyalty_tier,total_orders,lifetime_value,avg_order_value
CUST11,Anjali Singh,Delhi,Bronze,11,613170.0,55742.73
CUST04,Raj Patel,Hyderabad,Bronze,13,504357.0,38796.69
CUST13,Deepa Nair,Chennai,Silver,9,500078.0,55564.22
CUST14,Sanjay Gupta,Hyderabad,Silver,9,484274.0,53808.22
CUST12,Rahul Verma,Bengaluru,Bronze,11,468665.0,42605.91
CUST19,Divya Reddy,Surat,Silver,10,442470.0,44247.0
CUST05,Meena Nair,Pune,Gold,15,376365.0,25091.0
CUST16,Arun Krishnan,Kolkata,Bronze,14,344157.0,24582.64
CUST18,Suresh Babu,Jaipur,Bronze,10,309674.0,30967.4
CUST09,Lakshmi Iyer,Surat,Bronze,11,285261.0,25932.82


In [0]:
# 4d. Revenue by category
spark.sql("""
  CREATE OR REPLACE VIEW de_workspace26.ecommerce_badal.gold_revenue_by_category AS
  SELECT
    category,
    SUM(revenue)                AS total_revenue,
    COUNT(DISTINCT order_id)    AS total_orders,
    ROUND(AVG(revenue), 2)      AS avg_order_value
  FROM de_workspace26.ecommerce_badal.silver_orders
  GROUP BY category
  ORDER BY total_revenue DESC
""")

print("gold_revenue_by_category view created")
spark.sql("SELECT * FROM de_workspace26.ecommerce_badal.gold_revenue_by_category").display()


gold_revenue_by_category view created


category,total_revenue,total_orders,avg_order_value
Electronics,5102309.0,66,77307.71
Home,194599.0,35,5559.97
Sports,187569.0,50,3751.38
Clothing,64457.0,17,3791.59
Books,51397.0,32,1606.16


In [0]:
# 4e. Order status summary
spark.sql("""
  CREATE OR REPLACE VIEW de_workspace26.ecommerce_badal.gold_order_status_summary AS
  SELECT
    status,
    region,
    COUNT(order_id)   AS total_orders,
    SUM(revenue)      AS total_revenue
  FROM de_workspace26.ecommerce_badal.silver_orders
  GROUP BY status, region
  ORDER BY region, status
""")

print("gold_order_status_summary view created")
spark.sql("SELECT * FROM de_workspace26.ecommerce_badal.gold_order_status_summary")

gold_order_status_summary view created


DataFrame[status: string, region: string, total_orders: bigint, total_revenue: double]

In [0]:
spark.sql("SHOW VIEWS IN de_workspace26.ecommerce_badal LIKE 'gold_*'").display()

namespace,viewName,isTemporary,isMaterialized,isMetric
ecommerce_badal,gold_customer_ltv,false,false,false
ecommerce_badal,gold_monthly_revenue_by_region,false,false,false
ecommerce_badal,gold_order_status_summary,false,false,false
ecommerce_badal,gold_revenue_by_category,false,false,false
ecommerce_badal,gold_top_products,false,false,false


In [0]:
spark.sql("""
  CREATE OR REPLACE VIEW de_workspace26.ecommerce_badal.gold_top_products AS
  SELECT
    product_id,
    product_name,
    category,
    SUM(revenue)                                                     AS total_revenue,
    RANK() OVER (PARTITION BY category ORDER BY SUM(revenue) DESC)  AS revenue_rank
  FROM de_workspace26.ecommerce_badal.silver_orders
  GROUP BY product_id, product_name, category
""")

print("gold_top_products view created")

gold_top_products view created


In [0]:
spark.sql("""
  SELECT
    category,
    revenue_rank,
    product_name,
    total_revenue
  FROM de_workspace26.ecommerce_badal.gold_top_products
  ORDER BY category, revenue_rank
""").display()

category,revenue_rank,product_name,total_revenue
Books,1,Python Programming Book,51397.0
Clothing,1,Denim Jacket,64457.0
Electronics,1,Laptop Pro,2599948.0
Electronics,2,Smartphone,2239936.0
Electronics,3,Wireless Earbuds,262425.0
Home,1,Coffee Maker,110963.0
Home,2,Blender,52171.0
Home,3,Table Lamp,31465.0
Sports,1,Running Shoes,140736.0
Sports,2,Yoga Mat,46833.0


In [0]:

%sql SELECT
  year,
  month,
  region,
  total_revenue
FROM de_workspace26.ecommerce_badal.gold_monthly_revenue_by_region
ORDER BY year, month

year,month,region,total_revenue
2024,1,South,300663.0
2024,1,North,454868.0
2024,1,East,13993.0
2024,1,West,5097.0
2024,2,South,64170.0
2024,2,North,155588.0
2024,2,East,190961.0
2024,2,West,787970.0
2024,3,South,184080.0
2024,3,North,248977.0


In [0]:
%sql
SELECT
  revenue_rank,
  product_name,
  category,
  ROUND(total_revenue, 2) AS total_revenue
FROM de_workspace26.ecommerce_badal.gold_top_products
ORDER BY total_revenue DESC
LIMIT 10

revenue_rank,product_name,category,total_revenue
1,Laptop Pro,Electronics,2599948.0
2,Smartphone,Electronics,2239936.0
3,Wireless Earbuds,Electronics,262425.0
1,Running Shoes,Sports,140736.0
1,Coffee Maker,Home,110963.0
1,Denim Jacket,Clothing,64457.0
2,Blender,Home,52171.0
1,Python Programming Book,Books,51397.0
2,Yoga Mat,Sports,46833.0
3,Table Lamp,Home,31465.0


In [0]:
%sql
select * from de_workspace26.ecommerce_badal.orders version as of 0;


order_id,customer_id,product_id,order_date,quantity,unit_price,status,region,_ingested_at,_source_file
ORD0031,CUST12,PROD07,2024-04-15,4,34999.0,delivered,North,2026-04-08T05:13:20.923059Z,orders.csv
ORD0185,CUST03,PROD08,2024-06-28,3,699.0,delivered,North,2026-04-08T05:13:20.923059Z,orders.csv
ORD0060,CUST14,PROD04,2024-03-09,2,3499.0,delivered,North,2026-04-08T05:13:20.923059Z,orders.csv
ORD0077,CUST16,PROD10,2024-01-19,4,899.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv
ORD0159,CUST05,PROD07,2024-06-16,2,34999.0,placed,East,2026-04-08T05:13:20.923059Z,orders.csv
ORD0058,CUST12,PROD09,2024-04-14,5,499.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv
ORD0171,CUST10,PROD04,2024-04-08,4,3499.0,shipped,West,2026-04-08T05:13:20.923059Z,orders.csv
ORD0002,CUST15,PROD09,2024-02-01,4,499.0,placed,East,2026-04-08T05:13:20.923059Z,orders.csv
ORD0065,CUST04,PROD05,2024-02-15,5,2199.0,cancelled,East,2026-04-08T05:13:20.923059Z,orders.csv
ORD0148,CUST08,PROD02,2024-04-28,1,1499.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv


In [0]:
%sql
VACUUM de_workspace26.ecommerce_badal.orders RETAIN 100 HOURS;

path
s3://databricks-storage-7474660560922927/unity-catalog/7474660560922927/__unitystorage/catalogs/d5d0fd83-d5af-4053-943d-8af4a0b2c902/tables/96b02e74-44b4-458b-a021-d0b82dfcd5ca
